In [ ]:
import os
import time
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from pathlib import Path

try:
  os.chdir('../GitHub/FileCopy')
except:
  pass

path = os.getcwd()

SERVICE_ACCOUNT_FILE = "my-project-vscode-452201-249bdb033f8c.json"  # JSONファイルのパス
SCOPES = ["https://www.googleapis.com/auth/drive"]

LOCAL_ROOT = Path.cwd()

DRIVE_ROOT_ID = "14KlW9HehoZnVUhalRbHXlNRuTCKA5VN-"

MAX_SIZE_MB = 10

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

service = build("drive","v3",credentials=creds)

print(path)


/Users/takanohiroshi/Library/CloudStorage/OneDrive-個人用/Documenten/GitHub/FileCopy


In [ ]:
# Driceにフォルダを作成（存在しなけば）

def get_or_create_folder(name, parent_id):
    query = (
        f"name='{name}' and mimeType='application/vnd.google-apps.folder'"
        f" and '{parent_id}' in parents"
    )

    results = service.files().list(
        q=query,
        includeItemsFromAllDrives=True,
        supportsAllDrives=True
    ).execute()

    folders = results.get("files", [])
    if folders:
        return folders[0]["id"]

    file_metadata = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id]
    }

    folder = service.files().create(
        body=file_metadata,
        fields="id",
        supportsAllDrives=True
    ).execute()

    return folder["id"]

In [41]:
# Drice上のファイルの更新日時を取得
def get_drive_modified_time(file_id):
    file = service.files().get(
        fileId=file_id,
        fields="modifiedTime",
        supportsAllDrives=True
    ).execute()
    return file["modifiedTime"]

# Drive上に同名のファイルがあるか検索
def find_drive_file(name, parent_id):
    query = f"name='{name}' and '{parent_id}' in parents"

    results = service.files().list(
        q=query,
        includeItemsFromAllDrives=True,
        supportsAllDrives=True
    ).execute()

    files = results.get("files",[])
    return files[0] if files else None

# ファイルをアップロード（新しい場合のみ）
def upload_file(local_path, parent_id):
    size_mb = os.path.getsize(local_path) / (1024*1024)
    if size_mb > MAX_SIZE_MB:
        print("[SKIP]大きすぎる：{local_path}")
    name = os.path.basename(local_path)
    local_mtime = os.path.getmtime(local_path)

    drive_file = find_drive_file(name, parent_id)

    # Driceに同名ファイルがある場合->新しい方だけ反映
    if drive_file:
        drive_mtime_str = get_drive_modified_time(drive_file["id"])
        drive_mtime = time.mktime(time.strptime(drive_mtime_str, "%Y-%m-%dT%H:%M:%S.%fZ"))

        if drive_mtime >= local_mtime:
            print(f"[SKIP] Driveが新しい：{name}")
            return

        print(f"[UPDATE] Drive側を更新：{name}")
        media = MediaFileUpload(local_path, resumable=True)
        service.files().update(
            fileId=drive_file["id"],
            media_body=media,
            supportsAllDrives=True
        ).execute()
        return

    # Driveにファイルがない場合->新規アップロード
    print(f"[UPLOAD]新規アップロード：{name}")
    media = MediaFileUpload(local_path, resumable=True)
    file_metadata = {"name":name, "parents":[parent_id]}

    service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id",
        supportsAllDrives=True
    ).execute()

# 再起的にフォルダー構造をコピー
def copy_recursive(local_dir, drive_parent_id):
    for item in os.listdir(local_dir):
        local_path = os.path.join(local_dir, item)

        if os.path.isdir(local_path):
            new_drive_folder_id = get_or_create_folder(item, drive_parent_id)
            copy_recursive(local_path, new_drive_folder_id)
        else:
            upload_file(local_path, drive_parent_id)


In [42]:
copy_recursive(LOCAL_ROOT, DRIVE_ROOT_ID)



[UPLOAD]新規アップロード：my-project-vscode-452201-249bdb033f8c.json
